# $H_0$ sensitivity of the joint likelihood

Profile the joint training log-likelihood along $H_0$ with the other parameters held at their posterior medians. This shows **which datasets pull $H_0$ where** (SH0ES up, Planck-compressed down) — the tension is a real feature of the data, not an artifact.

Requires `data/` (shipped) — runs in a few minutes.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import numpy as np
import matplotlib.pyplot as plt
import lcdm_plus_s.bayesian_validation as bv

reg, theory, joint, post = bv.build_production_posterior(
    data_dir=str(Path.cwd().parent / 'data'), theory_nsteps=300)
summ = json.loads((Path.cwd().parent / 'output' / 'tables' / 'posterior_summary.json').read_text())
names = list(summ['parameters'])
med = {n: summ['parameters'][n]['median'] for n in names}
print('medians:', med)

In [ ]:
H0_grid = np.linspace(66, 76, 21)
per_probe = {getattr(lk, 'table2_key', lk.name): [] for lk in joint.likelihoods}
total = []
for h0 in H0_grid:
    theta = [h0, med['Omega_Lambda'], med['k'], med['t_crit']]
    tot = 0.0
    for lk in joint.likelihoods:
        v = lk.log_likelihood(theta)
        per_probe[getattr(lk, 'table2_key', lk.name)].append(v)
        tot += v
    total.append(tot)

fig, ax = plt.subplots(figsize=(8,5))
for key, vals in per_probe.items():
    v = np.asarray(vals); ax.plot(H0_grid, v - v.max(), label=key)
t = np.asarray(total); ax.plot(H0_grid, t - t.max(), 'k-', lw=2.5, label='joint')
ax.set_xlabel(r'$H_0$ [km/s/Mpc]'); ax.set_ylabel(r'$\Delta \ln L$')
ax.set_ylim(-50, 2); ax.legend(); ax.set_title('Per-dataset $H_0$ pulls (others fixed at posterior median)')
plt.show()